# `early_learning_predictors`: workbench

The definition lives in [`dataset.py`](dataset.py) and the evidence in [`README.md`](README.md). Edit `dataset.py`
and re-run a cell: `ds` picks up the change (the raw files are not read again unless `_load_raw` changed).

In [ ]:
from data_foundry.v2 import workbench

ds = workbench()
ds.df.head()

In [ ]:
ds.run_all_checks()["summary"]

In [ ]:
ds.preview_splits()

## Exploration

Migrated from the v1 notebook; adapt the cells where needed.

In [ ]:
df = ds.df.copy()

In [ ]:
print(f"Pri features: {len([col for col in df.columns if 'pri_' in col or '_pri' in col])}")
print(f"Pra features: {len([col for col in df.columns if 'pra_' in col or '_pra' in col])}")
print(f"Learning Programme Quality features: {len([col for col in df.columns if 'pqa_' in col or '_pqa' in col])}")
print(f"Observation features: {len([col for col in df.columns if 'obs_' in col or '_obs' in col])}")
print(f"Child features: {len([col for col in df.columns if 'child_' in col or '_child' in col])}")
print(f"Teacher features: {len([col for col in df.columns if 'teacher_' in col or '_teacher' in col])}")
print(f"Count per facility features: {len([col for col in df.columns if 'count_' in col])}")
print(f"Other features: {len(df.columns) - sum([len([col for col in df.columns if prefix in col]) for prefix in ['pri_', 'pra_', 'pqa_', 'teacher_', 'obs_', 'child_', 'count_']])}")

In [ ]:
# The Zindi train set is kept next to the shipped v1 notebook in datasets/beyond_iid/grouped/early_learning_predictors/.
zindi_train = pd.read_csv("../../../beyond_iid/grouped/early_learning_predictors/zindi_train.csv", low_memory=False)

# Matching the Zindi train set, to check whether we have the same data after dropping the columns
intersect_cols = list(set(df.columns).intersection(set(zindi_train.columns)))
print(f"{len(intersect_cols)} columns in common with Zindi train set.")

# New columns that are not in the Zindi train set, but are from a data subset collected independently of the target. start with pra_, pqa_, pri_, or obs_ )
pra_pqa_pri_obs_cols = [i for i in df.columns if i not in intersect_cols and ("pra_" in i or "pqa_" in i or "pri_" in i or "obs_" in i)]
print(f"{len(pra_pqa_pri_obs_cols)} columns are new and from  pra_, pqa_, pri_, or obs_ feature subsets.")

other_cols = [i for i in df.columns if i not in intersect_cols+pra_pqa_pri_obs_cols]
print(f"Remaining {len(other_cols)} columns that were not in the zindi challenge: {other_cols}")

print(f"Columns that are in the Zindi train set but not in our data: {[i for i in zindi_train.columns if i not in intersect_cols]}")

In [ ]:
# The data has many missing values, but they are not missing at random.
import matplotlib.pyplot as plt
# Missing value heatmap
plt.figure(figsize=(12, 6))
plt.imshow(df.isna(), aspect='auto', cmap='viridis')
plt.colorbar(label='Missing Value')
plt.title('Missing Values Heatmap')
plt.xlabel('Features')
plt.ylabel('Samples')
plt.show()

## Report

`ds.check()` runs the full pipeline and the bundle checks without saving and writes `README.md`. `ds.build()` also
saves the container (new UUID); only the curator runs it.

In [ ]:
result = ds.check()